In [ ]:
import arcpy
import os
import pandas as pd

def export_gdb_to_csv_by_alias(gdb_path, output_folder):
    """
    遍历指定File Geodatabase（GDB）中的所有要素类，包括要素数据集内的，
    并将每个要素类的属性表导出为一个单独的CSV文件。
    导出的CSV文件列名将使用字段的别名（Alias Name），如果没有别名则使用字段名。

    参数:
    - gdb_path: 字符串，File GDB (.gdb) 文件的完整路径。
    - output_folder: 字符串，用于存放CSV文件的输出文件夹路径。
    """
    # 检查 GDB 路径是否存在
    if not arcpy.Exists(gdb_path):
        print(f"错误: 找不到指定的地理数据库 '{gdb_path}'。请检查路径。")
        return

    # 检查输出文件夹是否存在，如果不存在则创建
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"已创建输出文件夹：'{output_folder}'")

    print(f"开始处理地理数据库： {gdb_path}")
    print("-" * 50)
    
    # 设置工作空间为 GDB 根目录
    arcpy.env.workspace = gdb_path
    
    # 1. 获取所有要素类，包括在要素数据集内的
    feature_classes = []
    
    # 获取根目录下的要素类
    feature_classes.extend(arcpy.ListFeatureClasses())
    
    # 获取要素数据集中的要素类
    datasets = arcpy.ListDatasets(feature_type='Feature')
    for dataset in datasets:
        arcpy.env.workspace = os.path.join(gdb_path, dataset)
        # 拼接完整路径，以便后续操作
        dataset_fcs = [os.path.join(dataset, fc) for fc in arcpy.ListFeatureClasses()]
        feature_classes.extend(dataset_fcs)
    
    # 恢复工作空间为 GDB 根目录
    arcpy.env.workspace = gdb_path
    
    if not feature_classes:
        print(f"在 {gdb_path} 中没有找到任何要素类。")
        return

    # 2. 遍历要素类并导出属性表
    for fc_path_relative in feature_classes:
        try:
            # 构建完整路径
            full_fc_path = os.path.join(gdb_path, fc_path_relative)
            
            # 获取要素类的名称，用于命名CSV文件
            if os.path.dirname(fc_path_relative):
                file_name = os.path.basename(fc_path_relative)
                output_csv = os.path.join(output_folder, f"{os.path.dirname(fc_path_relative)}_{file_name}.csv")
            else:
                file_name = os.path.basename(fc_path_relative)
                output_csv = os.path.join(output_folder, f"{file_name}.csv")

            print(f"正在导出要素类 '{fc_path_relative}' 的属性表...")

            # 获取所有字段信息
            fields = arcpy.ListFields(full_fc_path)
            # 排除'SHAPE@'字段，因为它不能直接写入CSV，并获取字段名列表
            field_names = [field.name for field in fields if field.name.lower() != 'shape' and field.name.lower() != 'shape@']
            
            # 获取字段别名列表，如果没有别名则使用字段名
            field_aliases = [field.aliasName if field.aliasName else field.name for field in fields if field.name.lower() != 'shape' and field.name.lower() != 'shape@']

            # 使用 arcpy.da.FeatureClassToNumPyArray 将属性表转为 NumPy 数组
            data = arcpy.da.FeatureClassToNumPyArray(full_fc_path, field_names)

            # 将 NumPy 数组转为 Pandas DataFrame
            df = pd.DataFrame(data)

            # 将DataFrame的列名修改为别名
            df.columns = field_aliases

            # 将 DataFrame 导出为 CSV 文件
            df.to_csv(output_csv, index=False, encoding='utf-8-sig')
            
            print(f"  - 导出成功到：'{output_csv}'")

        except arcpy.ExecuteError:
            print(f"  - 导出要素类 '{fc_path_relative}' 时出错：")
            print(arcpy.GetMessages(2))
        except Exception as e:
            print(f"  - 发生未知错误： {e}")
        
        print("-" * 50)
    
    print("所有要素类属性表导出完毕。")

if __name__ == '__main__':
    # ----------------------------------------------------
    # 用户需要修改以下参数
    # ----------------------------------------------------
    
    # 1. 指定你的 File Geodatabase (.gdb) 文件路径
    gdb_path = r"E:\成果库\CDData.gdb"  # 请修改为你的实际路径

    # 2. 指定输出文件夹路径
    output_folder_path = r"E:\成果库\表"  # 请修改为你的实际路径

    # ----------------------------------------------------
    # 调用函数，开始执行脚本
    # ----------------------------------------------------
    export_gdb_to_csv_by_alias(gdb_path, output_folder_path)